# 1. Install & Imports

In [1]:
!pip install xgboost imbalanced-learn --quiet

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score,
    accuracy_score, f1_score
)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

# 2. Load Dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/weather_data_only.csv')
df.head()

Mounted at /content/drive


,Timestamp,Temperature (°C),Humidity (%),Wind Speed (m/s),Rainfall (mm),Solar Irradiance (W/m²)
0,1/1/2020 0:00,28.993428,75.011269,1.053861,4.140513,185.892561
1,1/1/2020 0:15,27.723471,77.024015,1.085152,9.446997,281.782650
2,1/1/2020 0:30,29.295377,74.732958,3.363800,4.265813,328.942058
3,1/1/2020 0:45,31.046060,87.615995,2.539148,1.038103,336.407064
4,1/1/2020 1:00,27.531693,79.709858,1.366819,4.201393,205.494256


# 3. Feature Engineering

In [3]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

# Basic datetime features
df['hour']       = df['Timestamp'].dt.hour
df['minute']     = df['Timestamp'].dt.minute
df['day']        = df['Timestamp'].dt.day
df['month']      = df['Timestamp'].dt.month
df['year']       = df['Timestamp'].dt.year
df['dayofweek']  = df['Timestamp'].dt.dayofweek
df['dayofyear']  = df['Timestamp'].dt.dayofyear
df['weekofyear'] = df['Timestamp'].dt.isocalendar().week.astype(int)
df['quarter']    = df['Timestamp'].dt.quarter

# Cyclical encoding
df['hour_sin']      = np.sin(2 * np.pi * df['hour']      / 24)
df['hour_cos']      = np.cos(2 * np.pi * df['hour']      / 24)
df['month_sin']     = np.sin(2 * np.pi * df['month']     / 12)
df['month_cos']     = np.cos(2 * np.pi * df['month']     / 12)
df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)
df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)

# Sri Lanka monsoon seasons
def get_season(month):
    if month in [12, 1, 2]:         return 'NE_Monsoon'
    elif month in [3, 4]:            return 'First_InterMonsoon'
    elif month in [5, 6, 7, 8, 9]:  return 'SW_Monsoon'
    else:                            return 'Second_InterMonsoon'

le = LabelEncoder()
df['season_enc'] = le.fit_transform(df['month'].apply(get_season))
season_classes   = le.classes_

df['is_daytime'] = ((df['hour'] >= 6) & (df['hour'] <= 18)).astype(int)

# Lag features (1hr, 3hr, 24hr back)
weather_cols = [
    'Temperature (°C)', 'Humidity (%)', 'Wind Speed (m/s)',
    'Rainfall (mm)', 'Solar Irradiance (W/m²)'
]

for col in weather_cols:
    df[f'{col}_lag1']  = df[col].shift(1)
    df[f'{col}_lag3']  = df[col].shift(3)
    df[f'{col}_lag24'] = df[col].shift(24)

# Rolling mean features
for col in weather_cols:
    df[f'{col}_roll3']  = df[col].shift(1).rolling(3).mean()
    df[f'{col}_roll24'] = df[col].shift(1).rolling(24).mean()

df = df.dropna().reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["Timestamp"].min()} to {df["Timestamp"].max()}')

Dataset shape: (189864, 50)
Date range: 2020-01-01 06:00:00 to 2025-05-31 23:45:00


# 4. Create Target Labels

In [4]:
# 0 = Partly Cloudy | 1 = Overcast | 2 = Rainy
def assign_condition(row):
    if row['Rainfall (mm)'] >= 0.1:
        return 2
    elif row['Solar Irradiance (W/m²)'] >= 100 and row['Humidity (%)'] < 80:
        return 0
    else:
        return 1

CONDITION_NAMES = {0: 'Partly Cloudy', 1: 'Overcast', 2: 'Rainy'}
NUM_CLASSES     = 3

df['weather_condition'] = df.apply(assign_condition, axis=1)

print(df['weather_condition'].value_counts().rename(CONDITION_NAMES))

weather_condition
Rainy            185994
Partly Cloudy      1950
Overcast           1920
Name: count, dtype: int64


# 5. Define Features & Split

In [5]:
base_features = [
    'hour', 'minute', 'day', 'month', 'year',
    'dayofweek', 'dayofyear', 'weekofyear', 'quarter',
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'dayofyear_sin', 'dayofyear_cos',
    'dayofweek_sin', 'dayofweek_cos',
    'season_enc', 'is_daytime',
]

lag_features = []
for col in weather_cols:
    lag_features += [
        f'{col}_lag1', f'{col}_lag3', f'{col}_lag24',
        f'{col}_roll3', f'{col}_roll24'
    ]

ALL_FEATURES = base_features + weather_cols + lag_features

print(f'Total features: {len(ALL_FEATURES)}')

X = df[ALL_FEATURES]
y = df['weather_condition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')

Total features: 49
Train: 446385  |  Test: 37973


# 6. Hyperparameter Configuration

In [20]:

XGB_PARAMS = dict(
    n_estimators          = 600,
    max_depth             = 6,
    learning_rate         = 0.1,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 5,
    reg_alpha             = 0.1,
    reg_lambda            = 2.0,
    gamma                 = 0.5,
    early_stopping_rounds = 30,
    eval_metric           = 'mlogloss',
    objective             = 'multi:softprob',
    num_class             = NUM_CLASSES,
    tree_method           = 'hist',
    random_state          = 42,
    n_jobs                = -1,
)

# 7. Train XGBoost Classifier

In [21]:
clf = xgb.XGBClassifier(**XGB_PARAMS)

clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print(f'Best iteration: {clf.best_iteration}')

[0]	validation_0-mlogloss:0.98555
[50]	validation_0-mlogloss:0.01147
[100]	validation_0-mlogloss:0.00233
[149]	validation_0-mlogloss:0.00225
Best iteration: 119


# 8. Classification Report

In [22]:

y_pred       = clf.predict(X_test)
y_pred_proba = clf.predict_proba(X_test)

class_names = [CONDITION_NAMES[i] for i in range(NUM_CLASSES)]

print(classification_report(y_test, y_pred, target_names=class_names))

acc    = accuracy_score(y_test, y_pred)
f1_mac = f1_score(y_test, y_pred, average='macro')
f1_wtd = f1_score(y_test, y_pred, average='weighted')

try:
    auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
except Exception:
    auc = float('nan')

print(f'Accuracy        : {acc:.4f}')
print(f'Macro F1        : {f1_mac:.4f}')
print(f'Weighted F1     : {f1_wtd:.4f}')
print(f'ROC-AUC (macro) : {auc:.4f}')
print(f'Best round      : {clf.best_iteration}')

               precision    recall  f1-score   support

Partly Cloudy       0.99      0.97      0.98       390
     Overcast       0.98      0.98      0.98       384
        Rainy       1.00      1.00      1.00     37199

     accuracy                           1.00     37973
    macro avg       0.99      0.98      0.99     37973
 weighted avg       1.00      1.00      1.00     37973

Accuracy        : 0.9993
Macro F1        : 0.9869
Weighted F1     : 0.9993
ROC-AUC (macro) : 1.0000
Best round      : 119


# 10. Save Model

In [23]:

import os

save_dir = '/content/drive/MyDrive/weather_models'
os.makedirs(save_dir, exist_ok=True)

joblib.dump(clf, f'{save_dir}/xgb_weather_classifier.pkl')

joblib.dump({
    'features'        : ALL_FEATURES,
    'condition_names' : CONDITION_NAMES,
    'season_classes'  : list(season_classes),
    'metrics': {
        'accuracy'   : round(acc, 4),
        'f1_macro'   : round(f1_mac, 4),
        'f1_weighted': round(f1_wtd, 4),
        'roc_auc'    : round(auc, 4),
    }
}, f'{save_dir}/weather_classifier_meta.pkl')

print(f'Models saved to Google Drive -> {save_dir}')

Models saved to Google Drive -> /content/drive/MyDrive/weather_models


# 11. Prediction Function

In [24]:
def predict_condition(timestamp_str):
    dt = pd.to_datetime(timestamp_str)

    base_row = {
        'hour'          : dt.hour,
        'minute'        : dt.minute,
        'day'           : dt.day,
        'month'         : dt.month,
        'year'          : dt.year,
        'dayofweek'     : dt.dayofweek,
        'dayofyear'     : dt.dayofyear,
        'weekofyear'    : dt.isocalendar()[1],
        'quarter'       : dt.quarter,
        'hour_sin'      : np.sin(2 * np.pi * dt.hour      / 24),
        'hour_cos'      : np.cos(2 * np.pi * dt.hour      / 24),
        'month_sin'     : np.sin(2 * np.pi * dt.month     / 12),
        'month_cos'     : np.cos(2 * np.pi * dt.month     / 12),
        'dayofyear_sin' : np.sin(2 * np.pi * dt.dayofyear / 365),
        'dayofyear_cos' : np.cos(2 * np.pi * dt.dayofyear / 365),
        'dayofweek_sin' : np.sin(2 * np.pi * dt.dayofweek / 7),
        'dayofweek_cos' : np.cos(2 * np.pi * dt.dayofweek / 7),
        'season_enc'    : list(season_classes).index(get_season(dt.month)),
        'is_daytime'    : int(6 <= dt.hour <= 18),
    }

    mask = (df['hour'] == dt.hour) & (df['month'] == dt.month)
    lag_row = base_row.copy()
    for col in weather_cols:
        avg = df[mask][col].mean()
        lag_row[f'{col}_lag1']   = avg
        lag_row[f'{col}_lag3']   = avg
        lag_row[f'{col}_lag24']  = avg
        lag_row[f'{col}_roll3']  = avg
        lag_row[f'{col}_roll24'] = avg

    reg_feature_cols = base_features + lag_features
    reg_input = pd.DataFrame([lag_row])[reg_feature_cols]

    reg_models = {}
    for col in weather_cols:
        safe = (col.replace(' ', '_').replace('(', '').replace(')', '')
                   .replace('/', '').replace('°', 'deg')
                   .replace('%', 'pct').replace('²', '2'))
        reg_models[col] = joblib.load(
            f'/content/drive/MyDrive/weather_models/xgb_{safe}.pkl'
        )

    predicted_values = {}
    for col in weather_cols:
        val = float(reg_models[col].predict(reg_input)[0])
        if col in ['Rainfall (mm)', 'Solar Irradiance (W/m²)', 'Wind Speed (m/s)']:
            val = max(0.0, val)
        if col == 'Humidity (%)':
            val = min(100.0, max(0.0, val))
        predicted_values[col] = round(val, 2)

    clf_row = lag_row.copy()
    for col in weather_cols:
        clf_row[col] = predicted_values[col]

    clf_input  = pd.DataFrame([clf_row])[ALL_FEATURES]
    pred_class = int(clf.predict(clf_input)[0])
    condition  = CONDITION_NAMES[pred_class]

    return predicted_values, condition

# 12. Interactive Prediction

In [25]:

user_input = input('Enter a date and time (e.g., 2025-08-15 10:00): ')

try:
    values, condition = predict_condition(user_input)
    print(f'Timestamp : {user_input}')
    print('-' * 35)
    for col, val in values.items():
        print(f'  {col:<30} {val}')
    print(f'\n  Condition : {condition}')
except Exception as e:
    print(f'Error: {e}')
    print('Please use format: YYYY-MM-DD HH:MM')

Enter a date and time (e.g., 2025-08-15 10:00): 2026-09-24 12:30
Timestamp : 2026-09-24 12:30
-----------------------------------
  Temperature (°C)               27.42
  Humidity (%)                   82.3
  Wind Speed (m/s)               5.8
  Rainfall (mm)                  2.87
  Solar Irradiance (W/m²)        366.01

  Condition : Rainy
